In [2]:
%pip install scikit-learn joblib

     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.1/8.9 MB 991.0 kB/s eta 0:00:09
      --------------------------------------- 0.1/8.9 MB 939.4 kB/s eta 0:00:10
     - -------------------------------------- 0.2/8.9 MB 1.5 MB/s eta 0:00:06
     -- ------------------------------------- 0.5/8.9 MB 2.2 MB/s eta 0:00:04
     -- ------------------------------------- 0.6/8.9 MB 2.5 MB/s eta 0:00:04
     ---- ----------------------------------- 1.0/8.9 MB 3.5 MB/s eta 0:00:03
     ------ --------------------------------- 1.5/8.9 MB 4.3 MB/s eta 0:00:02
     ---------- ----------------------------- 2.3/8.9 MB 6.2 MB/s eta 0:00:02
     -------------- ------------------------- 3.2/8.9 MB 7.6 MB/s eta 0:00:01
     --------------- ------------------------ 3.4/8.9 MB 8.0 MB/s eta 0:00:01
     --------------- ------------------------ 3.4/8.9 MB 8.0 MB/s eta 0:00

In [3]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load Artifacts from Notebook 3
train_df = pd.read_csv("artifacts/03_train.csv")
val_df = pd.read_csv("artifacts/03_val.csv")
test_df = pd.read_csv("artifacts/03_test.csv")

# 2. Function to Extract Raw Features (Time & Basic Features)
def extract_features(df):
    data = df.copy()
    
    # Parse Timestamps
    purchase_dt = pd.to_datetime(data['order_purchase_timestamp'], format='mixed')
    estimated_dt = pd.to_datetime(data['order_estimated_delivery_date'], format='mixed')
    
    # Feature Engineering
    data['purchase_dayofweek'] = purchase_dt.dt.dayofweek
    data['purchase_hour'] = purchase_dt.dt.hour
    data['estimated_delivery_duration_days'] = (estimated_dt - purchase_dt).dt.total_seconds() / (24 * 3600)
    
    return data

print("Extracting features...")
train_feat = extract_features(train_df)
val_feat = extract_features(val_df)
test_feat = extract_features(test_df)

# 3. Define Feature Columns
numeric_features = [
    'total_price', 
    'total_freight', 
    'total_items_count', 
    'total_payment_value', 
    'payment_installments_max',
    'purchase_dayofweek',
    'purchase_hour',
    'estimated_delivery_duration_days'
]

categorical_features = ['customer_state']

target_col = 'is_late'

# 4. Build Preprocessing Pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 5. Fit ONLY on Train, Transform on Train, Val, Test
print("Fitting preprocessor on training data...")
X_train = preprocessor.fit_transform(train_feat[numeric_features + categorical_features])
y_train = train_feat[target_col].values

X_val = preprocessor.transform(val_feat[numeric_features + categorical_features])
y_val = val_feat[target_col].values

X_test = preprocessor.transform(test_feat[numeric_features + categorical_features])
y_test = test_feat[target_col].values

# Extract Feature Names
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_cols = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_cols

print(f"\nFinal Features count: {len(all_feature_names)}")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

# 6. Save Artifacts
os.makedirs("artifacts/features", exist_ok=True)

# Save Fitted Preprocessor Object
joblib.dump(preprocessor, "artifacts/features/preprocessor.joblib")

# Save Processed Arrays and Feature List
np.savez_compressed(
    "artifacts/features/processed_data.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)

with open("artifacts/features/feature_names.txt", "w") as f:
    for item in all_feature_names:
        f.write(f"{item}\n")

print("\nArtifacts successfully saved to 'artifacts/features/':")
print(" - preprocessor.joblib")
print(" - processed_data.npz")
print(" - feature_names.txt")

Extracting features...
Fitting preprocessor on training data...

Final Features count: 35
X_train shape: (77176, 35), y_train shape: (77176,)
X_val shape:   (9647, 35), y_val shape:   (9647,)
X_test shape:  (9647, 35), y_test shape:  (9647,)

Artifacts successfully saved to 'artifacts/features/':
 - preprocessor.joblib
 - processed_data.npz
 - feature_names.txt
